# Notebook 03 — Leakage-safe split

**ClinicalShield v2 · Revision pipeline**

## Purpose
Partition the dataset so that no source document appears in more than one partition, then
**measure** whether the length confound is actually gone rather than asserting it.

## The two questions this notebook answers

**Reviewer 1:** *"clarify whether different encoded variants derived from the same original
sample were separated across the training, validation, and test sets to avoid potential
data leakage."*

Answered structurally. `group_id` was assigned in NB01 and inherited by every chunk and
every encoded variant in NB02. `StratifiedGroupKFold` keeps each group whole, and Cell 4
asserts it — the notebook halts rather than producing a contaminated split.

**Reviewer 4:** *"Severe source/length confounding exists: PubMed benign samples have a
median 2,648 words versus 20 for Deepset attacks."*

Answered by measurement. Cell 6 trains a classifier on **word count alone**. If length
still carries signal, that probe will find it. The same probe is run against a
reconstruction of v1's length distribution, which quantifies how much of v1's reported
accuracy could have come from length rather than detection.

That contrast is the number for the response letter: a claim becomes a measurement.

## Reviewer points addressed
- **R1-e** encoded variants confined to one partition, asserted not claimed
- **R4-6** length confound measured, with a v1 counterfactual for contrast
- **R3-Q7** alternative split ratios recorded in config, swept in NB11

## Runtime
CPU. Seconds.

In [ ]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
CONFIG = {
    "SEED": 42,
    "SPLIT": {"train": 0.70, "val": 0.15, "test": 0.15},

    # Alternative ratios swept in NB11 to answer R3-Q7. Listed here so the
    # primary choice is visibly a choice, not an unexamined default.
    "ALT_SPLITS": [(0.80, 0.10, 0.10), (0.60, 0.20, 0.20)],

    "USE_DRIVE":      True,
    "DRIVE_DIR":      "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB": True,
    "GITHUB_REPO":    "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":  "v2-revision",
    "GIT_USER_NAME":  "Adu-Boahene Bright",
    "GIT_USER_EMAIL": "baduboahene@st.knust.edu.gh",
}
SEED = CONFIG["SEED"]
print("split:", CONFIG["SPLIT"], "| seed:", SEED)


In [ ]:
# ── Cell 2 · Environment and NB02 inputs ──────────────────────────────────────
import sys, os, json, random, hashlib, subprocess, shutil, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

random.seed(SEED); np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT, REPO_DIR = None, None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"]); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive :", DRIVE_ROOT)
    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR = Path("/content") / repo_name
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            r = subprocess.run(["git","clone","-q",
                "https://" + tok + "@github.com/" + CONFIG["GITHUB_REPO"] + ".git",
                str(REPO_DIR)], capture_output=True, text=True)
            print("clone :", "ok" if r.returncode == 0 else r.stderr[:200])
        except Exception as e:
            print("clone skipped:", type(e).__name__)
    else:
        print("clone : already present")
    if REPO_DIR.exists():
        for k, v in [("user.name", CONFIG["GIT_USER_NAME"]),
                     ("user.email", CONFIG["GIT_USER_EMAIL"])]:
            subprocess.run(["git","-C",str(REPO_DIR),"config",k,v], check=False)
        subprocess.run(["git","-C",str(REPO_DIR),"checkout","-q",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
        subprocess.run(["git","-C",str(REPO_DIR),"pull","-q","origin",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
    ROOT = REPO_DIR if REPO_DIR.exists() else Path("/content")
else:
    ROOT = Path.cwd()
    while not (ROOT/".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT/".git").exists():
        ROOT = Path.cwd()

DIRS = {"dataset": ROOT/"data"/"dataset", "stats": ROOT/"data"/"stats",
        "splits": ROOT/"data"/"splits"}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("root  :", ROOT)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def restore(rel_path, label):
    local = ROOT / rel_path
    if local.exists():
        return local, "repo"
    if DRIVE_ROOT:
        src = DRIVE_ROOT / rel_path
        if src.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, local)
            return local, "drive"
    drive_msg = str(DRIVE_ROOT / rel_path) if DRIVE_ROOT else "(drive not mounted)"
    raise FileNotFoundError("\n" + label + " not found. Looked in:\n  repo : "
        + str(local) + "\n  drive: " + drive_msg + "\nRun NB02 first.")

p_ds, src_ds = restore("data/dataset/attack_dataset.jsonl", "attack_dataset.jsonl")
p_st, src_st = restore("data/stats/attack_stats.json",      "attack_stats.json")

dataset   = read_jsonl(p_ds)
nb02_stats = json.load(open(p_st))

exp_n = nb02_stats["counts"]["chunks"]
assert len(dataset) == exp_n, "row count %d != NB02 stats %d" % (len(dataset), exp_n)
exp_adv = nb02_stats["counts"]["adversarial"]
act_adv = sum(d["label"] for d in dataset)
assert act_adv == exp_adv, "adversarial %d != NB02 stats %d" % (act_adv, exp_adv)

print("")
print("attack_dataset.jsonl from " + src_ds + "  " + str(len(dataset)) + " chunks")
print("groups      : %d" % len({d["group_id"] for d in dataset}))
print("adversarial : %d (%.1f%%)" % (act_adv, 100*act_adv/len(dataset)))
print("NB02 inputs verified")


## The split

Two passes. The first holds out 30% of groups, the second halves that into validation and
test — both group-aware, so a document cannot be divided.

`StratifiedGroupKFold` balances the label distribution across folds while keeping groups
intact. Because each group is one source document containing a mix of benign and poisoned
chunks, perfect stratification is not achievable; Cell 5 reports the residual drift.

In [ ]:
# ── Cell 3 · Read the split assigned in NB02 ──────────────────────────────────
# NB02 now assigns partitions, because payload templates must be drawn from a
# partition-specific pool. NB03's role is verification, which is stronger: the
# notebook that creates a split should not also be the one that certifies it.

assert all("split" in d for d in dataset), "dataset has no split field — re-run NB02"

parts = {s: [d for d in dataset if d["split"] == s] for s in ["train", "val", "test"]}
split_of = {i: d["split"] for i, d in enumerate(dataset)}
groups = np.array([d["group_id"] for d in dataset])
y = np.array([d["label"] for d in dataset])

print("partition    chunks   groups   adversarial")
for s in ["train", "val", "test"]:
    p = parts[s]
    print("  %-8s %6d   %6d   %5d (%.1f%%)"
          % (s, len(p), len({d["group_id"] for d in p}),
             sum(d["label"] for d in p), 100*sum(d["label"] for d in p)/len(p)))


## Leakage verification

**This cell is a gate, not a report.** If any group or source document spans partitions,
it raises and the pipeline stops. A split that silently leaks is worse than no split,
because every downstream number would be inflated and nothing would say so.

Two independent checks: one on `group_id`, one on `parent_doc_id`. The second is redundant
if the first holds, which is the point — the guarantee should not depend on a single
field being correct.

In [ ]:
# ── Cell 4 · Leakage verification (hard gate) ─────────────────────────────────
# This is the direct answer to Reviewer 1's question about encoded variants of
# the same source document appearing in different partitions.

g2s = defaultdict(set)
for d in dataset:
    g2s[d["group_id"]].add(d["split"])

spanning = {g: s for g, s in g2s.items() if len(s) > 1}
print("CHECK 1 — no group spans partitions")
print("  groups total         : %d" % len(g2s))
print("  groups spanning >1   : %d" % len(spanning))
if spanning:
    for g, s in list(spanning.items())[:5]:
        print("    %s -> %s" % (g, sorted(s)))
print("  -> %s" % ("PASS" if not spanning else "FAIL"))
assert not spanning, "group leakage detected across partitions"

# parent documents must also be confined, which follows from group_id but is
# checked independently so the guarantee does not rest on one field
d2s = defaultdict(set)
for d in dataset:
    d2s[d["parent_doc_id"]].add(d["split"])
doc_span = {k: v for k, v in d2s.items() if len(v) > 1}
print("")
print("CHECK 2 — no source document spans partitions")
print("  documents spanning >1: %d  -> %s" % (len(doc_span), "PASS" if not doc_span else "FAIL"))
assert not doc_span, "source document leakage detected"

# ── CHECK 3 — no attack template spans partitions ────────────────────────────
# The first build passed the group check and failed this one: 47 of 48 templates
# appeared in all three partitions. Module 2 reached 99.70% accuracy by
# memorising them. Group disjointness alone is not sufficient.

tpl_split = defaultdict(set)
for d in dataset:
    if d.get("template_id"):
        tpl_split[d["template_id"]].add(d["split"])
tpl_spanning = {t: s for t, s in tpl_split.items() if len(s) > 1}
tpl_test_only = [t for t, s in tpl_split.items() if s == {"test"}]

print("")
print("CHECK 3 — no attack template spans partitions")
print("  templates used            : %d" % len(tpl_split))
print("  templates spanning >1     : %d" % len(tpl_spanning))
print("  templates unique to test  : %d" % len(tpl_test_only))
print("  -> %s" % ("PASS" if not tpl_spanning else "FAIL"))
assert not tpl_spanning, "template leakage across partitions"

# every chunk assigned exactly once
assert len(split_of) == len(dataset), "unassigned chunks"
counts = Counter(d["split"] for d in dataset)
assert sum(counts.values()) == len(dataset)
print("")
print("CHECK 4 — assignment completeness")
print("  chunks assigned      : %d/%d -> PASS" % (sum(counts.values()), len(dataset)))


## Stratification quality

Per-category results in the paper are only meaningful if each category is actually present
in the test partition. Anything under 20 test samples is flagged, because a rate computed
on fewer than that carries a confidence interval too wide to report.

In [ ]:
# ── Cell 5 · Stratification quality ───────────────────────────────────────────
# Every attack type must be represented in test, or per-category results in the
# paper would rest on whatever happened to land there.

print("label balance by partition")
for s in ["train","val","test"]:
    p = parts[s]
    a = sum(d["label"] for d in p)
    print("  %-6s adversarial %.1f%%" % (s, 100*a/len(p)))

overall = 100*sum(y)/len(y)
drift = max(abs(100*sum(d["label"] for d in parts[s])/len(parts[s]) - overall)
            for s in ["train","val","test"])
print("  max drift from overall %.1f%% : %.2f pp -> %s"
      % (overall, drift, "PASS" if drift < 3 else "REVIEW"))

for field in ["vector", "category", "encoding"]:
    print("")
    print("%s coverage in test" % field)
    all_vals = sorted({str(d[field]) for d in dataset if d["label"] == 1})
    tst = Counter(str(d[field]) for d in parts["test"] if d["label"] == 1)
    for v in all_vals:
        n = tst.get(v, 0)
        flag = "" if n >= 20 else "   <-- thin"
        print("  %-28s %5d%s" % (v, n, flag))
    missing = [v for v in all_vals if tst.get(v, 0) == 0]
    if missing:
        print("  MISSING FROM TEST: %s" % missing)


## Confound diagnostics

The heart of the notebook.

A probe classifier sees **only the word count** — no text, no tokens, no features of any
kind beyond length. Its AUC is a direct measure of how much the label can be predicted
from length alone.

- **AUC near 0.50** means length carries no signal and the confound is gone.
- **AUC near 1.00** means the classifier could score well without reading anything.

The same probe is then run on a reconstruction of v1's distribution, built from the medians
Reviewer 4 reported. This is the honest way to address that criticism: not "we fixed it",
but "here is what it was, here is what it is now, measured the same way".

In [ ]:
# ── Cell 6 · Confound diagnostics (the R4-6 evidence) ─────────────────────────
# Reviewer 4: "Severe source/length confounding exists: PubMed benign samples
# have a median 2,648 words versus 20 for Deepset attacks."
#
# A claim that this is fixed is weak. A measurement is not. If a classifier
# given ONLY the word count can separate the classes, the confound is alive.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.dummy import DummyClassifier

def length_only_probe(train_part, test_part, label=""):
    Xtr = np.array([[d["word_count"]] for d in train_part], dtype=float)
    ytr = np.array([d["label"] for d in train_part])
    Xte = np.array([[d["word_count"]] for d in test_part], dtype=float)
    yte = np.array([d["label"] for d in test_part])

    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    prob = clf.predict_proba(Xte)[:, 1]
    auc  = roc_auc_score(yte, prob)
    acc  = accuracy_score(yte, clf.predict(Xte))

    base = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
    base_acc = accuracy_score(yte, base.predict(Xte))
    return {"auc": float(auc), "accuracy": float(acc),
            "majority_baseline": float(base_acc), "n_test": int(len(yte))}

probe_v2 = length_only_probe(parts["train"], parts["test"])
print("LENGTH-ONLY PROBE — v2 corpus")
print("  A model given nothing but word count, nothing else.")
print("  AUC                : %.4f   (0.50 = no signal)" % probe_v2["auc"])
print("  accuracy           : %.4f" % probe_v2["accuracy"])
print("  majority baseline  : %.4f" % probe_v2["majority_baseline"])
verdict_v2 = "PASS" if probe_v2["auc"] < 0.60 else "FAIL - confound still present"
print("  -> %s" % verdict_v2)

# ── counterfactual: what the same probe would score on v1's design ───────────
# Reconstructed from the medians Reviewer 4 reported. This quantifies how much
# of v1's headline accuracy could have come from length alone.
rng_cf = np.random.default_rng(SEED)
n_cf = 2000
benign_cf = rng_cf.lognormal(np.log(2648), 0.6, n_cf)
attack_cf = rng_cf.lognormal(np.log(20),   0.6, n_cf)
cf_train = ([{"word_count": w, "label": 0} for w in benign_cf[:1500]] +
            [{"word_count": w, "label": 1} for w in attack_cf[:1500]])
cf_test  = ([{"word_count": w, "label": 0} for w in benign_cf[1500:]] +
            [{"word_count": w, "label": 1} for w in attack_cf[1500:]])
probe_v1 = length_only_probe(cf_train, cf_test)

print("")
print("LENGTH-ONLY PROBE — v1 design, reconstructed (2,648 vs 20 word medians)")
print("  AUC                : %.4f" % probe_v1["auc"])
print("  accuracy           : %.4f" % probe_v1["accuracy"])
print("")
print("  Interpretation: under v1's corpus design, word count alone separates")
print("  the classes almost perfectly. Under v2, it carries no signal.")
print("  v1 AUC %.4f  ->  v2 AUC %.4f" % (probe_v1["auc"], probe_v2["auc"]))

# ── source probe: both classes now come from the same corpus ─────────────────
# In v1, benign came from PubMed and attacks from Deepset, so source and label
# were confounded. Here every chunk originates from openFDA, so this is
# structural rather than something to tune away.
src_fields = {d.get("section") for d in dataset}
print("")
print("SOURCE PROBE")
print("  benign and adversarial chunks share one corpus (openFDA) : True")
print("  label sections represented                               : %d" % len(src_fields))
print("  -> source/label confounding removed by construction")


## Persist

`attack_dataset_split.jsonl` carries the split assignment on every row, so no downstream
notebook has to re-derive it — and none of them can disagree about it.

`group_assignments.json` is small and goes into git as the permanent leakage record. A
reviewer can verify the partitioning independently without access to the corpus.

In [ ]:
# ── Cell 7 · Save splits ──────────────────────────────────────────────────────
def write_jsonl(p, rows):
    with open(p, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return p

ds_path = write_jsonl(DIRS["dataset"] / "attack_dataset_split.jsonl", dataset)

# group -> partition map, small enough to live in git as the leakage record
group_map = {}
for d in dataset:
    group_map[d["group_id"]] = d["split"]
with open(DIRS["splits"] / "group_assignments.json", "w") as f:
    json.dump(group_map, f, indent=2, sort_keys=True)

split_stats = {
    "notebook": "03_leakage_safe_split",
    "seed": SEED,
    "method": "StratifiedGroupKFold on group_id (one group = one source document)",
    "ratios": CONFIG["SPLIT"],
    "partitions": {
        s: {
            "chunks": len(parts[s]),
            "groups": len({d["group_id"] for d in parts[s]}),
            "adversarial": int(sum(d["label"] for d in parts[s])),
            "benign": int(len(parts[s]) - sum(d["label"] for d in parts[s])),
            "adversarial_pct": float(100 * sum(d["label"] for d in parts[s]) / len(parts[s])),
        } for s in ["train", "val", "test"]
    },
    "leakage_checks": {
        "groups_spanning_partitions": len(spanning),
        "documents_spanning_partitions": len(doc_span),
        "templates_spanning_partitions": len(tpl_spanning),
        "templates_unique_to_test": len(tpl_test_only),
        "all_chunks_assigned": len(split_of) == len(dataset),
        "note": ("Group disjointness alone proved insufficient: an earlier build "
                 "kept documents apart but shared all payload templates across "
                 "partitions, and the classifier memorised them."),
    },
    "confound_probes": {
        "length_only_v2": probe_v2,
        "length_only_v1_counterfactual": probe_v1,
        "verdict": verdict_v2,
    },
    "test_coverage": {
        f: dict(Counter(str(d[f]) for d in parts["test"] if d["label"] == 1))
        for f in ["vector", "category", "encoding"]
    },
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}
with open(DIRS["stats"] / "split_stats.json", "w") as f:
    json.dump(split_stats, f, indent=2)

if IN_COLAB and DRIVE_ROOT:
    for sub in ["dataset", "stats", "splits"]:
        dst = DRIVE_ROOT / "data" / sub
        dst.mkdir(parents=True, exist_ok=True)
        for f_ in (ROOT / "data" / sub).glob("*"):
            shutil.copy2(f_, dst / f_.name)
    print("mirrored to Drive")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    keep = ["data/stats/split_stats.json", "data/splits/group_assignments.json"]
    subprocess.run(["git", "-C", str(ROOT), "add", "-f"] + keep, check=False)
    st = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        msg = ("NB03: leakage-safe split, length-only AUC %.4f" % probe_v2["auc"])
        subprocess.run(["git", "-C", str(ROOT), "commit", "-q", "-m", msg], check=False)
        pr = subprocess.run(["git", "-C", str(ROOT), "push", "-q", "origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push:", "ok" if pr.returncode == 0 else pr.stderr[:200])

print("written: %s (%.1f MB)" % (ds_path.name, ds_path.stat().st_size/1024/1024))
print("written: group_assignments.json (%d groups)" % len(group_map))


## Summary

**Paste the output below into the chat.**

The number I will be checking first is the length-only AUC. If it is above 0.60, the
confound survived and NB04 must not proceed.

In [ ]:
# ── Cell 8 · NB03 SUMMARY — paste this output ─────────────────────────────────
print("=" * 62)
print("NB03 — LEAKAGE-SAFE SPLIT")
print("=" * 62)
print("method : StratifiedGroupKFold on group_id")
print("seed   : %d" % SEED)
print("")
print("partition    chunks   groups   adversarial")
for s in ["train", "val", "test"]:
    p = parts[s]
    print("  %-8s %6d   %6d   %5d (%.1f%%)"
          % (s, len(p), len({d["group_id"] for d in p}),
             sum(d["label"] for d in p),
             100*sum(d["label"] for d in p)/len(p)))
print("")
print("LEAKAGE CHECKS (R1-e)")
print("  groups spanning partitions    : %d  -> %s"
      % (len(spanning), "PASS" if not spanning else "FAIL"))
print("  documents spanning partitions : %d  -> %s"
      % (len(doc_span), "PASS" if not doc_span else "FAIL"))
print("  templates spanning partitions : %d  -> %s"
      % (len(tpl_spanning), "PASS" if not tpl_spanning else "FAIL"))
print("  templates unique to test      : %d" % len(tpl_test_only))
print("  all chunks assigned           : %s" % (len(split_of) == len(dataset)))
print("")
print("CONFOUND DIAGNOSTIC (R4-6)")
print("  length-only AUC, v2 corpus        : %.4f" % probe_v2["auc"])
print("  length-only accuracy, v2          : %.4f" % probe_v2["accuracy"])
print("  majority-class baseline           : %.4f" % probe_v2["majority_baseline"])
print("  length-only AUC, v1 counterfactual: %.4f" % probe_v1["auc"])
print("  verdict                           : %s" % verdict_v2)
print("")
print("TEST COVERAGE (adversarial only)")
for f in ["vector", "category", "encoding"]:
    cnt = Counter(str(d[f]) for d in parts["test"] if d["label"] == 1)
    print("  %s:" % f)
    for k, v in sorted(cnt.items(), key=lambda x: -x[1]):
        print("    %-28s %5d" % (k, v))
print("")
print("PROBE SET BUDGET (NB04, Option C)")
print("  test chunks available as hosts : %d" % len(parts["test"]))
print("=" * 62)
print("NB03 COMPLETE — ready for NB04 (Module 1 + encoding probe set)")
print("=" * 62)
